In [1]:
print("hello gatinha")

hello gatinha


In [4]:
import json
import pandas as pd

# 1. Carregar os dados de temperatura e umidade
df_temp = pd.read_csv("sensor_temperature_24_09_2026.csv")
df_hum = pd.read_csv("sensor_humidity_24_09_2026.csv")

# 2. Carregar o arquivo de dispositivos
with open("devices.txt", "r", encoding="utf-8") as f:
    devices_data = json.load(f)

df_devices = pd.DataFrame(devices_data)[["id", "name"]]

# 3. Garantir compatibilidade de tipos nas chaves de junção
df_temp["sensor_id"] = df_temp["sensor_id"].astype(str)
df_hum["sensor_id"] = df_hum["sensor_id"].astype(str)
df_devices["id"] = df_devices["id"].astype(str)

# 4. Juntar leituras de temperatura e umidade
# Faz o merge pelas colunas comuns (sensor_id, measured_in e environment)
df_merged = pd.merge(
    df_temp,
    df_hum,
    on=["measured_in", "sensor_id", "environment"],
    how="inner",
    suffixes=("_temp", "_hum"),
)

# Se houver sensor_model redundante, mantém um deles limpo
if "sensor_model_temp" in df_merged.columns:
    df_merged["sensor_model"] = df_merged["sensor_model_temp"]
    df_merged.drop(
        columns=["sensor_model_temp", "sensor_model_hum"], inplace=True
    )

# 5. Adicionar o nome do dispositivo fazendo o match por sensor_id == id
df_results = pd.merge(
    df_merged, df_devices, left_on="sensor_id", right_on="id", how="left"
)

# Remove a coluna 'id' duplicada vinda do devices
df_results.drop(columns=["id"], inplace=True)

# 6. Reorganizar as colunas para melhor legibilidade
colunas_finais = [
    "measured_in",
    "sensor_id",
    "name",
    "sensor_model",
    "environment",
    "temperature",
    "humidity",
]
# Mantém apenas as colunas existentes na lista
df_results = df_results[
    [col for col in colunas_finais if col in df_results.columns]
]

# 7. Salvar o arquivo final
df_results.to_csv("results.csv", index=False)
print("Arquivo 'results.csv' gerado com sucesso!")

Arquivo 'results.csv' gerado com sucesso!


In [ ]:
import numpy as np


# Abordagem 1: Regra Baseada em ASHRAE 55 / NR-17
def classificar_conforto(row):
    t = row["temperature"]
    h = row["humidity"]

    if pd.isna(t) or pd.isna(h):
        return "Indeterminado"

    # Critérios de conforto
    if 20.0 <= t <= 26.0 and 30.0 <= h <= 65.0:
        return "Confortavel"
    elif t < 20.0:
        return "Frio"
    elif t > 26.0 and h > 65.0:
        return "Muito Quente e Umido"
    elif t > 26.0:
        return "Quente"
    elif h < 30.0:
        return "Ar Seco"
    else:
        return "Desconfortavel"


df_results["thermal_comfort"] = df_results.apply(classificar_conforto, axis=1)

# Abordagem 2: Índice Numérico de Thom (DI)
t = df_results["temperature"]
rh = df_results["humidity"]

df_results["thom_di"] = t - 0.55 * (1 - 0.01 * rh) * (t - 14.5)
df_results["thom_comfort"] = pd.cut(
    df_results["thom_di"],
    bins=[-np.inf, 21, 24, 27, np.inf],
    labels=["Confortavel", "Desconforto Leve", "Desconfortavel", "Severo"],
)

In [5]:
import json
import pandas as pd

with open("devices.txt", "r", encoding="utf-8") as f:
    devices = json.load(f)

df = pd.DataFrame(devices)

# Exibe total e lista de ID + Name
print(f"Total de dispositivos: {len(df)}")
print(f"IDs únicos: {df['id'].nunique()}")
print(df[["id", "name"]].sort_values("id").to_string(index=False))

# Se quiser filtrar apenas sensores de temperatura/umidade disponíveis (available == True)
sensores_ativos = df[
    (df["available"] == True)
    & (df["name"].str.contains("TEMP|TH", case=False, na=False))
]
print(
    f"\nSensores de temperatura/umidade ativos ({len(sensores_ativos)} encontrados):"
)
print(sensores_ativos[["id", "name"]].to_string(index=False))

Total de dispositivos: 65
IDs únicos: 65
   id                                name
    1                               TEMP2
    2            Ar Samsung 1006 esquerda
    4              Prototipo 1008 com LDR
    6                   Prototipo Cluster
   12                            TCL 1008
   13                      Ar Daikin 1008
   15             Protoboard 1006 com LDR
   16                           Teste OTA
   18                          MAC Camera
   22                Samsung Meio Cluster
   23          Ar Samsung Direita Vilalba
   24                     1006-TEMPHUM-05
   45                                 AC1
   46                  1008EXT-TEMPHUM-06
   47                        Teste CoAP 3
   48                     1006-TEMPHUM-07
   49                               TEMP3
  884                      Sensor Externo
  885              1006CLUSTER-TEMPHUM-04
  886                                 NaN
  887                     1006-TEMPHUM-09
  888                  1006EXT-TEMP

In [7]:
import json

# Nome do arquivo de entrada e de saída
arquivo_entrada = "devices.txt"  # Altere para o nome do seu arquivo
arquivo_saida = "sensores_atualizados.json"

# 1. Carrega o arquivo JSON/TXT
with open(arquivo_entrada, "r", encoding="utf-8") as f:
    sensores = json.load(f)

# 2. Adiciona as novas chaves em cada elemento da lista
for sensor in sensores:
    sensor["location_x"] = 0
    sensor["location_y"] = 0
    sensor["location_confirmed"] = False

# 3. Salva a lista atualizada em um novo arquivo
with open(arquivo_saida, "w", encoding="utf-8") as f:
    json.dump(sensores, f, indent=2, ensure_ascii=False)

print(
    f"Processamento concluído! {len(sensores)} sensores atualizados em '{arquivo_saida}'."
)

Processamento concluído! 65 sensores atualizados em 'sensores_atualizados.json'.


In [8]:
import json

arquivo_entrada = "sensores_atualizados.json"  # Altere para o nome do seu arquivo
arquivo_saida = "sensores_env1.json"

# 1. Carrega os dados do arquivo
with open(arquivo_entrada, "r", encoding="utf-8") as f:
    sensores = json.load(f)

# 2. Filtra apenas os itens com id_enviroment == 1
sensores_env1 = [s for s in sensores if s.get("id_enviroment") == 1]

# 3. Salva a nova lista filtrada
with open(arquivo_saida, "w", encoding="utf-8") as f:
    json.dump(sensores_env1, f, indent=2, ensure_ascii=False)

print(
    f"Filtro concluído! {len(sensores_env1)} sensores salvos em '{arquivo_saida}'."
)

Filtro concluído! 16 sensores salvos em 'sensores_env1.json'.


In [12]:
import json
import pandas as pd

# 1. Carregar o arquivo original de dispositivos
with open("sensores_env1.json", "r", encoding="utf-8") as f:
    devices_data = json.load(f)

df_devices = pd.DataFrame(devices_data)

# 2. Filtrar onde 'name' OU 'description' contenham 'TEMPHUM'
filtro_temphum = df_devices["name"].fillna("").str.contains(
    "TEMPHUM", case=False
) | df_devices["description"].fillna("").str.contains("TEMPHUM", case=False)

df_filtered = df_devices[filtro_temphum].sort_values(by="id")

# 3. Converter o DataFrame para uma lista de dicionários
resultado_json = df_filtered.to_dict(orient="records")

# 4. Salvar a saída em formato JSON formatado
with open("devices_temphum.json", "w", encoding="utf-8") as f:
    json.dump(resultado_json, f, indent=2, ensure_ascii=False)

print(
    f"Arquivo 'devices_temphum.json' gerado com sucesso! Total: {len(resultado_json)} sensores."
)

Arquivo 'devices_temphum.json' gerado com sucesso! Total: 8 sensores.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.interpolate import RBFInterpolator

# ==========================================
# 1. ARQUIVOS E DIMENSÕES DA SALA
# ==========================================
CSV_FILE = "sensor_temperature_24_09_2026.csv"
JSON_FILE = "sensores_meus.json"

X_MIN, X_MAX = 0.0, 8.0
Y_MIN, Y_MAX = 0.0, 5.0
TIME_STEP = "1h"

# ==========================================
# 2. CARREGAR COORDENADAS DO JSON
# ==========================================
with open(JSON_FILE, "r", encoding="utf-8") as f:
    sensors_list = json.load(f)

sensor_coords = {}
for s in sensors_list:
    s_id = str(s["id"])
    x = float(s["location_x"])
    y = float(s["location_y"])
    
    if X_MIN <= x <= X_MAX and Y_MIN <= y <= Y_MAX:
        sensor_coords[s_id] = (x, y)

# ==========================================
# 3. CARREGAR E ALINHAR MEDIDAS DO CSV
# ==========================================
df = pd.read_csv(CSV_FILE)
df['measured_in'] = pd.to_datetime(df['measured_in'])
df['sensor_id'] = df['sensor_id'].astype(str)

pivot_df = df.pivot_table(
    index='measured_in',
    columns='sensor_id',
    values='temperature',
    aggfunc='mean'
)

# Filtra colunas válidas ANTES de qualquer interpolação
active_sensors = [s for s in pivot_df.columns if s in sensor_coords]

if len(active_sensors) < 3:
    raise ValueError(f"Poucos sensores ativos mapeados ({len(active_sensors)}). Mínimo de 3 necessário.")

# Alinha e preenche somente os sensores ativos
resampled_df = pivot_df[active_sensors].resample(TIME_STEP).mean().ffill().bfill()
sensor_pts = np.array([sensor_coords[s] for s in active_sensors])

# ==========================================
# 4. MALHA ESPACIAL E INTERPOLAÇÃO
# ==========================================
grid_x, grid_y = np.meshgrid(
    np.linspace(X_MIN, X_MAX, 120),
    np.linspace(Y_MIN, Y_MAX, 100)
)
flat_grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

def interpolate_temperatures(temps):
    rbf = RBFInterpolator(sensor_pts, temps, kernel='linear')
    return rbf(flat_grid).reshape(grid_x.shape)

# ==========================================
# 5. PRÉ-CÁLCULO E ANIMAÇÃO
# ==========================================
print(f"Pré-calculando {len(resampled_df)} frames...")
frames_data = [
    interpolate_temperatures(resampled_df.iloc[i].values)
    for i in range(len(resampled_df))
]

fig, ax = plt.subplots(figsize=(10, 6))

# Limites baseados no resultado final das interpolações
vmin = min(f.min() for f in frames_data)
vmax = max(f.max() for f in frames_data)

heatmap = ax.imshow(
    frames_data[0],
    origin='lower',
    extent=[X_MIN, X_MAX, Y_MIN, Y_MAX],
    cmap='coolwarm',
    vmin=vmin,
    vmax=vmax,
    interpolation='bicubic'
)

cbar = fig.colorbar(heatmap, ax=ax)
cbar.set_label('Temperatura (°C)', fontsize=11)

ax.scatter(sensor_pts[:, 0], sensor_pts[:, 1], c='black', edgecolors='white', s=70, zorder=5)
for s_id, (sx, sy) in zip(active_sensors, sensor_pts):
    ax.text(sx + 0.1, sy + 0.1, s_id, fontsize=8, fontweight='bold', color='black', zorder=6)

ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(Y_MIN, Y_MAX)
ax.set_xlabel('X (metros)')
ax.set_ylabel('Y (metros)')
title_text = ax.set_title("", fontsize=12)

def update_frame(frame_idx):
    heatmap.set_data(frames_data[frame_idx])
    current_time = resampled_df.index[frame_idx]
    title_text.set_text(f"Horário: {current_time.strftime('%Y-%m-%d %H:%M:%S')}")
    return heatmap, title_text

ani = animation.FuncAnimation(
    fig,
    update_frame,
    frames=len(frames_data),
    interval=20,
    blit=True
)

plt.tight_layout()
plt.show()

Pré-calculando 2730092 frames...
